<a href="https://colab.research.google.com/github/mdkamrulhasan/data_mining_kdd/blob/main/notebooks/regression_diabetes_knn_HP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### What will we cover today ?

This is a companion to the *Linear Regression with Regularization* notebook. We use the **same dataset and the same train/test split**, but replace linear regression with a **k-Nearest Neighbors (kNN) regressor** (***diabetes severity prediction***).

1.   **kNN Regression** (default setting, k = 5)
2.   **Effect of k**: a very small k (k = 1) vs. a very large k (k = 50)
3.   **Comparing model performances**
4.   **Hyper-parameter Optimization - kNN Regression** (number of neighbors *k* and the weighting scheme)

**Parallel with the regression notebook:** in linear regression the regularization strength (alpha / lambda) controls model complexity. In kNN, the number of neighbors **k** plays that role:

| | Low complexity (may underfit) | High complexity (may overfit) |
|---|---|---|
| Ridge / Lasso | large alpha | small alpha |
| kNN | large k | small k |

[Diabetes Dataset](https://www4.stat.ncsu.edu/~boos/var.select/diabetes.html)

**Summary of the dataset:**

*   **Features:** *age, sex, bmi, bp, s1, s2, s3, s4, s5, s6*
*   **Target (y)**: A quantitative measure to track the progression of diabetes.

**Note:** Some features are directly identifiable, say, age/sex/bmi, while some others are derived features (for more details, read the data description provided through the link above).

---

---

## Loading necessary python packages

In [33]:
# database related package(s)
from sklearn import datasets

# data processing packages
import pandas as pd
import numpy as np

# Regression modeling package(s) (sklearn)
from sklearn.neighbors import KNeighborsRegressor
# Linear regression is only used as a reference point at the very end
from sklearn.linear_model import LinearRegression

# model evaluation related packages
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error

# visualization
import plotly.express as px
import plotly.graph_objects as go

## Loading data and some preprocessing

In [2]:
# Load the dataset
df = datasets.load_diabetes(as_frame=True)
df.keys()

dict_keys(['data', 'target', 'frame', 'DESCR', 'feature_names', 'data_filename', 'target_filename', 'data_module'])

In [3]:
# Separating features and labels dataframes
features_df, labels_df = df.data, df.target

In [4]:
# Looking at a sample of features
features_df.head()

,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6
0,0.038076,0.050680,0.061696,0.021872,-0.044223,-0.034821,-0.043401,-0.002592,0.019907,-0.017646
1,-0.001882,-0.044642,-0.051474,-0.026328,-0.008449,-0.019163,0.074412,-0.039493,-0.068332,-0.092204
2,0.085299,0.050680,0.044451,-0.005670,-0.045599,-0.034194,-0.032356,-0.002592,0.002861,-0.025930
3,-0.089063,-0.044642,-0.011595,-0.036656,0.012191,0.024991,-0.036038,0.034309,0.022688,-0.009362
4,0.005383,-0.044642,-0.036385,0.021872,0.003935,0.015596,0.008142,-0.002592,-0.031988,-0.046641


In [5]:
# plotting the (bmi, target) pair values
fig = px.scatter(x=features_df['bmi'], y=labels_df.values,
                 labels={'x': 'bmi', 'y': 'disease progression'})
fig.show()

Patients with similar BMI tend to have similar disease progression. kNN exploits exactly this idea: **to predict a new patient, find the k most similar patients in the training data and average their targets.** Unlike linear regression, kNN does not assume the relationship is a straight line.

## Modeling

Extracting features and labels as numpy matrices

In [6]:
X, y = features_df.values, labels_df.values

Splitting data into train, test splits

(Same proportion and random seed as the linear-regression notebook, so the results are directly comparable.)

In [80]:
# test data amount (in terms of proportion)
TEST_PROP = 0.5
# Random number seed; important for experiment reproducibility
RANDOM_SEED = 0

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_PROP, random_state=RANDOM_SEED)
X_train.shape, y_train.shape, X_test.shape, y_test.shape

((221, 10), (221,), (221, 10), (221,))

**A note on feature scaling:** kNN is distance-based, so features on larger scales dominate the distance. The sklearn diabetes features are *already* mean-centered and scaled to a common range (look at `features_df.head()` above), so we can use them directly in Sections 1-3. In the hyper-parameter optimization section we still put a `StandardScaler` inside a pipeline, because that is the habit you need for almost every other dataset.

## [kNN Regression](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsRegressor.html) (k = 5, the sklearn default)

Model Instantiation

In [81]:
# Create a kNN regression object (k = 5 neighbors, sklearn's default)
regr_knn = KNeighborsRegressor(n_neighbors=5)

Model Training

For kNN, "training" simply means **storing** the training data; all the work happens at prediction time when neighbors are searched.

In [82]:
# Train (i.e., memorize) the model using the training set
regr_knn.fit(X_train, y_train)

KNeighborsRegressor()

Making Predictions and evaluation (on the training data)

- just checking how good the model fit was on the training data.

In [83]:
# Making predictions (training dataset)
y_pred = regr_knn.predict(X_train)
# Estimating the Mean Squared Error (MSE)
mse_train = mean_squared_error(y_train, y_pred)
print("MSE (train data): %.2f" % mse_train)

MSE (train data): 2533.32


Making Predictions and evaluation (on the test data)

- This is more interesting metric as we are reporting on unseen data (by the model)

In [84]:
# Making predictions (test dataset)
y_pred = regr_knn.predict(X_test)
# Estimating the Mean Squared Error (MSE)
mse_test = mean_squared_error(y_test, y_pred)
print("MSE (test data): %.2f" % mse_test)

MSE (test data): 3719.89


In [85]:
knn5_results = pd.DataFrame({
  'model': ['knn (k=5)'],
  'train_err': [round(mse_train, 2)],
  'test_err': [round(mse_test, 2)]
})

Note(s):

*   Compare the train and test errors. Why is the gap here larger than for linear regression?
*   Hint: when predicting a *training* point, that point is its own nearest neighbor (distance 0), so it always contributes to its own prediction.

### Looking inside a kNN prediction

Linear models let us inspect coefficients (`coef_`). kNN has no coefficients; instead, we can look at **which neighbors** were used for a prediction.

In [86]:
# Take the first test patient
x_new = X_test[0:1]

# Indices (into X_train) and distances of its 5 nearest neighbors
distances, indices = regr_knn.kneighbors(x_new)

neighbors = pd.DataFrame({
    'neighbor_index': indices[0],
    'distance': distances[0].round(4),
    'neighbor_target': y_train[indices[0]]
})
print(neighbors)
print("\nAverage of neighbor targets (= kNN prediction): %.2f" % y_train[indices[0]].mean())
print("Model prediction:                               %.2f" % regr_knn.predict(x_new)[0])
print("True target for this patient:                   %.2f" % y_test[0])

   neighbor_index  distance  neighbor_target
0              15    0.0455            336.0
1             169    0.0665            131.0
2              21    0.0823            263.0
3             127    0.0869            263.0
4             111    0.0928            151.0

Average of neighbor targets (= kNN prediction): 228.80
Model prediction:                               228.80
True target for this patient:                   321.00


## Effect of k: a very small k (k = 1)

In [87]:
# Create a kNN regression object with only ONE neighbor
regr_knn1 = KNeighborsRegressor(n_neighbors=1)
regr_knn1.fit(X_train, y_train)

# Prediction and error estimation (training data)
y_pred = regr_knn1.predict(X_train)
mse_train = mean_squared_error(y_train, y_pred)
print("MSE (train data): %.2f" % mse_train)

# Prediction and error estimation (test data)
y_pred = regr_knn1.predict(X_test)
mse_test = mean_squared_error(y_test, y_pred)
print("MSE (test data): %.2f" % mse_test)

# Storing results in a dataframe
knn1_results = pd.DataFrame({
  'model': ['knn (k=1)'],
  'train_err': [round(mse_train, 2)],
  'test_err': [round(mse_test, 2)]
})

MSE (train data): 0.00
MSE (test data): 6110.96


With k = 1 every training point is predicted by itself, so the training error is **exactly 0**, yet the test error is the worst so far. This is **overfitting**: the model memorizes the noise in the training data.

## Effect of k: a very large k (k = 50)

In [88]:
# Create a kNN regression object with 50 neighbors
regr_knn50 = KNeighborsRegressor(n_neighbors=50)
regr_knn50.fit(X_train, y_train)

# Prediction and error estimation (training data)
y_pred = regr_knn50.predict(X_train)
mse_train = mean_squared_error(y_train, y_pred)
print("MSE (train data): %.2f" % mse_train)

# Prediction and error estimation (test data)
y_pred = regr_knn50.predict(X_test)
mse_test = mean_squared_error(y_test, y_pred)
print("MSE (test data): %.2f" % mse_test)

# Storing results in a dataframe
knn50_results = pd.DataFrame({
  'model': ['knn (k=50)'],
  'train_err': [round(mse_train, 2)],
  'test_err': [round(mse_test, 2)]
})

MSE (train data): 3531.55
MSE (test data): 3326.74


With k = 50 each prediction averages over almost a quarter of the training set, so predictions are pulled toward the overall mean. The train and test errors are now close to each other (low variance), but the training error itself is high (high bias): the model is starting to **underfit**.

## Comparing model performances

In [89]:
results = pd.concat([knn1_results, knn5_results, knn50_results], axis=0)

fig = go.Figure([
    go.Bar(x=results.model, y=results.train_err, name='Training error'),
    go.Bar(x=results.model, y=results.test_err, name='Test error')
])
fig.update_layout(title="Model comparison (effect of k)", yaxis_title="MSE")
fig.update_layout(legend=dict(x=0.05, y=0.999))
fig.show()

# Questions for you

*   How do training and test errors change as k goes from 1 to 5 to 50?
*   Which of these three values of k is overfitting and which is underfitting?
*   We picked k = 1, 5, 50 by hand. How could we choose k in a principled way **without** looking at the test set?



---

---


# Hyper-parameter Optimization

In [90]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV

## Pipeline Definition

We put the scaler and the kNN model in one `Pipeline`. During cross-validation the scaler is then re-fit on each training fold only, so no information from the validation fold leaks into the scaling.

In [91]:
knn_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsRegressor())
])

## Hyperparameter Grid Definition

* `n_neighbors` = number of neighbors **k** (plays the role alpha / lambda played for Ridge)
* `weights` = how neighbors are combined:
    * `"uniform"`: simple average of the k neighbors
    * `"distance"`: closer neighbors get a larger weight (weight = 1 / distance)

Note the `knn__` prefix: it tells the pipeline that the parameter belongs to the step named `"knn"`.

In [92]:
param_grid = {
    "knn__n_neighbors": list(range(1, 51)),     # k = 1, 2, ..., 50
    "knn__weights": ["uniform", "distance"]
}

## Applying Five-fold Cross Validation on the Training Set

In [93]:
grid_search = GridSearchCV(
    estimator=knn_pipeline,
    param_grid=param_grid,
    scoring="neg_mean_squared_error",
    cv=5,
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('scaler', StandardScaler()),
                                       ('knn', KNeighborsRegressor())]),
             n_jobs=-1,
             param_grid={'knn__n_neighbors': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11,
                                              12, 13, 14, 15, 16, 17, 18, 19,
                                              20, 21, 22, 23, 24, 25, 26, 27,
                                              28, 29, 30, ...],
                         'knn__weights': ['uniform', 'distance']},
             scoring='neg_mean_squared_error')

## Displaying Grid Search Results

In [94]:
print("Best k:      ", grid_search.best_params_["knn__n_neighbors"])
print("Best weights:", grid_search.best_params_["knn__weights"])
print("Best CV MSE:  %.2f" % -grid_search.best_score_)

# Best trained model (already re-fit on the full training set)
cv_model = grid_search.best_estimator_

Best k:       13
Best weights: distance
Best CV MSE:  3350.17


## Visualizing the Search: Cross-validated MSE vs. k

This is the kNN version of a validation curve. Small k (left) overfits, large k (right) underfits; the best k sits in between.

In [95]:
cv_results = pd.DataFrame(grid_search.cv_results_)
cv_results["cv_mse"] = -cv_results["mean_test_score"]
cv_results["k"] = cv_results["param_knn__n_neighbors"].astype(int)
cv_results["weights"] = cv_results["param_knn__weights"].astype(str)

fig = px.line(cv_results.sort_values("k"), x="k", y="cv_mse", color="weights",
              markers=True,
              labels={"k": "number of neighbors (k)", "cv_mse": "5-fold CV MSE"},
              title="Cross-validated MSE vs. k")
best_k = grid_search.best_params_["knn__n_neighbors"]
fig.add_vline(x=best_k, line_dash="dash",
              annotation_text=f"best k = {best_k}", annotation_position="top")
fig.show()

In [96]:
cv_model.get_params()

{'memory': None,
 'steps': [('scaler', StandardScaler()),
  ('knn', KNeighborsRegressor(n_neighbors=13, weights='distance'))],
 'transform_input': None,
 'verbose': False,
 'scaler': StandardScaler(),
 'knn': KNeighborsRegressor(n_neighbors=13, weights='distance'),
 'scaler__copy': True,
 'scaler__with_mean': True,
 'scaler__with_std': True,
 'knn__algorithm': 'auto',
 'knn__leaf_size': 30,
 'knn__metric': 'minkowski',
 'knn__metric_params': None,
 'knn__n_jobs': None,
 'knn__n_neighbors': 13,
 'knn__p': 2,
 'knn__weights': 'distance'}

## Evaluating Model Performance with the Hyperparameters Chosen through the HP Search

In [97]:
# Prediction and error estimation (training data)
y_pred = cv_model.predict(X_train)
mse_train = mean_squared_error(y_train, y_pred)
print("MSE (train data): %.2f" % mse_train)

# Prediction and error estimation (test data)
y_pred = cv_model.predict(X_test)
mse_test = mean_squared_error(y_test, y_pred)
print("MSE (test data): %.2f" % mse_test)

# Storing results in a dataframe
knn_cv_results = pd.DataFrame({
  'model': ['knn-cv (k=%d, %s)' % (best_k, grid_search.best_params_["knn__weights"])],
  'train_err': [round(mse_train, 2)],
  'test_err': [round(mse_test, 2)]
})

MSE (train data): 0.00
MSE (test data): 3065.53


**Why is the training error 0 again?** With `weights="distance"`, a training point is its own neighbor at distance 0, which gets an (effectively) infinite weight. So a training MSE of 0 here does **not** mean overfitting by itself: that is why we judge the model by the cross-validated and test errors, never by the training error.

## Putting it all together: kNN with varios k values

For reference we refit plain linear regression on the same split (this reproduces the result from the linear-regression notebook).

In [98]:
all_results = pd.concat([knn1_results, knn5_results, knn50_results,
                         knn_cv_results], axis=0)
print(all_results.to_string(index=False))

fig = go.Figure([
    go.Bar(x=all_results.model, y=all_results.test_err, name='Test error')
])
fig.update_layout(title="Model comparison: kNN variants",
                  yaxis_title="MSE")
fig.update_layout(legend=dict(x=0.05, y=0.999))
fig.show()

                  model  train_err  test_err
              knn (k=1)       0.00   6110.96
              knn (k=5)    2533.32   3719.89
             knn (k=50)    3531.55   3326.74
knn-cv (k=13, distance)       0.00   3065.53


# Questions for you

*   How much did hyper-parameter optimization improve kNN's test error compared with the hand-picked k = 5?
*   After tuning, kNN and linear regression end up with similar test error on this dataset. Recall the BMI scatter plot at the top: why might a simple linear model already be hard to beat here?
*   Which model would you prefer to deploy, and why? Think about interpretability (coefficients vs. neighbors), prediction speed, and how each would behave with 100x more patients.
*   **Try it yourself:** add `"knn__p": [1, 2]` to `param_grid` (Manhattan vs. Euclidean distance). Does the best model change?